In [ ]:
import cv2
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import os
import random
import torch
from torch.utils.data import DataLoader, TensorDataset
import kagglehub
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:

path=kagglehub.dataset_download("andrewmvd/leukemia-classification")

Using Colab cache for faster access to the 'leukemia-classification' dataset.


In [ ]:
DATA_DIR = "../kaggle/input/leukemia-classification/C-NMC_Leukemia"
TRAIN_DIR = os.path.join(DATA_DIR,"training_data")
TEST_DIR = os.path.join(DATA_DIR,"testing_data")
VALIDATION_DIR = os.path.join(DATA_DIR,"validation_data")

# --- 1. Configuration ---
# Update this path to where your C-NMC dataset is stored
CATEGORIES = ['hem', 'all'] # 'hem' = 0 (Healthy), 'all' = 1 (Leukemia)
IMG_SIZE = 64 # Resize to 64x64 to prevent memory crashes

def process_image(img_path):
    """
    Reads, converts to grayscale, resizes, and flattens one image.
    Returns flattened array.
    """
    try:
        img_array = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        resized_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
        flattened_array = resized_array.flatten()
        return flattened_array
    except Exception:
        return None  # skip unreadable images
# --- 2. Data Loading & Flattening ---
def load_data(DATA_DIR):
    data = []
    labels = []

    for category in CATEGORIES:
        path = os.path.join(DATA_DIR, category)
        class_num = CATEGORIES.index(category)

        img_paths = [os.path.join(path, f) for f in os.listdir(path)]
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = {executor.submit(process_image, img_path): img_path for img_path in img_paths}
            for future in as_completed(futures):
                flattened_array = future.result()
                if flattened_array is not None:
                    data.append(flattened_array)
                    labels.append(class_num)

    return (np.array(data), np.array(labels))
FOLD0DIR = os.path.join(TRAIN_DIR, "fold_0")
FOLD1DIR = os.path.join(TRAIN_DIR, "fold_1")
FOLD2DIR = os.path.join(TRAIN_DIR, "fold_2")
folds_data = {1:load_data(FOLD0DIR), 2:load_data(FOLD1DIR), 3:load_data(FOLD2DIR)}
print("Loading and flattening images...")

Loading and flattening images...


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Lists to hold the metrics for each of the 3 folds
accuracies = []
precisions = []
recalls = []
f1_scores = [] # NEW: Added F1 list

# --- 3. The Manual 3-Fold Loop ---
for test_fold_num in range(1, 4):
    print(f"\n--- Running Fold {test_fold_num} ---")

    # 1. Isolate the Test Data
    X_test, y_test = folds_data[test_fold_num]

    # 2. Combine the other two folds for Training Data
    X_train_list = []
    y_train_list = []

    for train_fold_num in range(1, 4):
        if train_fold_num != test_fold_num:
            X_train_list.append(folds_data[train_fold_num][0])
            y_train_list.append(folds_data[train_fold_num][1])

    # Stack the arrays vertically/horizontally to create single training matrices
    X_train = np.vstack(X_train_list)
    y_train = np.concatenate(y_train_list)

    # 3. Scale the Data (Crucial: Fit ONLY on the training data to prevent leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. Train the Model
    model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
    model.fit(X_train_scaled, y_train)

    # 5. Evaluate
    predictions = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, predictions)

    # We use average='weighted' to match the Transformer pipeline perfectly
    prec = precision_score(y_test, predictions, average='weighted')
    rec = recall_score(y_test, predictions, average='weighted')
    f1 = f1_score(y_test, predictions, average='weighted') # NEW: F1 Calculation

    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1) # NEW: Store F1

    print(f"  Accuracy:  {acc * 100:.2f}%")
    print(f"  Precision: {prec * 100:.2f}%")
    print(f"  Recall:    {rec * 100:.2f}%")
    print(f"  F1 Score:  {f1 * 100:.2f}%")

# --- 4. Final Average Scores ---
print("\n=== Final Average Scores across all 3 Folds ===")
print(f"Mean Accuracy:  {np.mean(accuracies) * 100:.2f}%")
print(f"Mean Precision: {np.mean(precisions) * 100:.2f}%")
print(f"Mean Recall:    {np.mean(recalls) * 100:.2f}%")
print(f"Mean F1 Score:  {np.mean(f1_scores) * 100:.2f}%")


--- Running Fold 1 ---
  Accuracy:  71.68%
  Precision: 72.43%
  Recall:    71.68%
  F1 Score:  71.99%

--- Running Fold 2 ---
  Accuracy:  75.65%
  Precision: 75.39%
  Recall:    75.65%
  F1 Score:  75.51%

--- Running Fold 3 ---
  Accuracy:  71.01%
  Precision: 71.05%
  Recall:    71.01%
  F1 Score:  71.03%

=== Final Average Scores across all 3 Folds ===
Mean Accuracy:  72.78%
Mean Precision: 72.96%
Mean Recall:    72.78%
Mean F1 Score:  72.84%
